In [ ]:
import pandas as pd
import polars as pl

In [ ]:
embedding_path = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/"
meta_df = (
    pd.read_csv(embedding_path + "ckd_embedding_full_v3_icd_stage_filter/meta_v3.csv", sep="$")
    .merge(
        pd.read_csv(embedding_path + "CKD-supplemental_pull.rpt", sep="$")
        , on='PatientID', how='left'
    )
)
meta_df['date'] = pd.to_datetime(meta_df['date'])
meta_df['DOB'] = pd.to_datetime(meta_df['DOB'])


In [ ]:
meta_df.head()

In [ ]:

meta_df['max_stage'] = meta_df.groupby('PatientID')['max_stage'].bfill().ffill()

In [ ]:

meta_df['CKD_stage_numeric'] = meta_df.groupby('PatientID')['CKD_stage_numeric'].bfill().ffill()

In [ ]:
meta_df.head()

In [ ]:
def unique_patient_ckd_counts(df, column_):
    # Select only the necessary columns and drop duplicate rows based on PatientID
    # to ensure each patient is counted only once for their CKD stage.
    # column_ = "max_stage" # "CKD_stage_numeric"
    unique_patients_ckd = df[['PatientID', column_]].drop_duplicates(subset=['PatientID'])

    # Count the occurrences of each CKD stage among these unique patients
    ckd_stage_counts = unique_patients_ckd[column_].value_counts()

    return ckd_stage_counts.sort_index()

In [ ]:
df1 = unique_patient_ckd_counts(meta_df, "EthnoRacialCategory")
df1

In [ ]:
import os
out_path1 = os.path.join("results_files", "EthnoRacialCategory_bar.csv")
out_path1

In [ ]:
df1.to_csv(out_path1)

In [ ]:
ax = df1.plot.barh()
ax.set_title('Racial Category Counts')
ax.set_xlabel('Category')
# xlim = ax.get_xlim()
# ax.set_xlim(xlim[0] - 0.5, xlim[1] + 0.5)
# ax.tick_params(axis='x', rotation=45) 

ax.set_ylabel('Count')

In [ ]:
ax = unique_patient_ckd_counts(meta_df, "Sex").plot.barh()


ax.set_title('Sex Counts')
ax.set_xlabel('Category')

ax.set_ylabel('Count')

In [ ]:
df3 = unique_patient_ckd_counts(meta_df, "max_stage")
df3

In [ ]:
out_path3 = os.path.join("results_files", "max_ckd_stage_bar.csv")
out_path3

In [ ]:
df3.to_csv(out_path3)

In [ ]:
ax = df3.plot.bar(rot=0)


ax.set_title('CKD Stage Counts')
ax.set_xlabel('Max CKD Stage')
ax.set_ylabel('Count')

In [ ]:
meta_df['max_stage'].unique()

In [ ]:
meta_df.columns